# StyleGAN2-ADA Animefaces 512 One-Click Colab

Runtime -> Change runtime type -> GPU, then run the single cell below.

Everything used during training is kept under `/content`. After training finishes, the run directory is zipped and copied to `MyDrive/alibaba`.

In [ ]:
#@title StyleGAN2-ADA Animefaces 512 一键训练
REPO_URL = "https://github.com/DavidFeng0718/AnimeGAN.git" #@param {type:"string"}
BRANCH = "stylegan512" #@param {type:"string"}
WORKDIR = "/content/AnimeGAN" #@param {type:"string"}
CONTENT_ROOT = "/content/stylegan2_ada_animefaces_512" #@param {type:"string"}
DRIVE_BACKUP_DIR = "/content/drive/MyDrive/alibaba" #@param {type:"string"}
KAGGLE_USERNAME = "" #@param {type:"string"}
KAGGLE_KEY = "" #@param {type:"string"}
KIMG = 5000 #@param {type:"integer"}
BATCH_SIZE = 64 #@param {type:"integer"}
SNAP = 50 #@param {type:"integer"}
AUGPIPE = "color" #@param ["color"]
RESUME = "latest-if-available" #@param ["noresume", "latest-if-available", "latest"]

import json
import os
from pathlib import Path
import shlex
import shutil
import subprocess
import sys
from datetime import datetime

def run(cmd, cwd=None, env=None):
    print(f"$ {cmd}")
    subprocess.run(cmd, shell=True, check=True, cwd=cwd, env=env)

def q(value):
    return shlex.quote(str(value))

def get_colab_secret(name):
    try:
        from google.colab import userdata
        return userdata.get(name) or ""
    except Exception:
        return ""

def configure_kaggle(username, key, content_root):
    kaggle_dir = content_root / "kaggle"
    kaggle_json = kaggle_dir / "kaggle.json"
    username = (username or "").strip() or get_colab_secret("KAGGLE_USERNAME")
    key = (key or "").strip() or get_colab_secret("KAGGLE_KEY")
    kaggle_dir.mkdir(parents=True, exist_ok=True)
    if username and key:
        kaggle_json.write_text(json.dumps({"username": username, "key": key}), encoding="utf-8")
    elif Path("/content/kaggle.json").exists():
        shutil.copy2("/content/kaggle.json", kaggle_json)
    elif not kaggle_json.exists():
        from google.colab import files
        print("Upload kaggle.json from Kaggle Account settings.")
        uploaded = files.upload()
        if "kaggle.json" not in uploaded:
            raise RuntimeError("kaggle.json was not uploaded and Kaggle credentials are missing.")
        kaggle_json.write_bytes(uploaded["kaggle.json"])
    kaggle_json.chmod(0o600)
    return kaggle_dir

workdir = Path(WORKDIR).resolve()
content_root = Path(CONTENT_ROOT).resolve()
drive_backup_dir = Path(DRIVE_BACKUP_DIR).resolve()
dataset_zip = content_root / "datasets" / "animefaces_512.zip"
download_dir = content_root / "kaggle_download"
run_root = content_root / "runs"
config_dir = content_root / "configs"
content_root.mkdir(parents=True, exist_ok=True)
dataset_zip.parent.mkdir(parents=True, exist_ok=True)
download_dir.mkdir(parents=True, exist_ok=True)
run_root.mkdir(parents=True, exist_ok=True)
config_dir.mkdir(parents=True, exist_ok=True)

from google.colab import drive
drive.mount("/content/drive")

run("nvidia-smi || true")

if workdir.exists():
    run(f"git -C {q(workdir)} fetch origin {q(BRANCH)}")
    run(f"git -C {q(workdir)} checkout {q(BRANCH)}")
    run(f"git -C {q(workdir)} pull --ff-only origin {q(BRANCH)}")
else:
    run(f"git clone --branch {q(BRANCH)} --single-branch {q(REPO_URL)} {q(workdir)}")

project_dir = workdir / "StyleGAN2-ADA"
run(f"{q(sys.executable)} -m pip install --upgrade pip")
run(f"{q(sys.executable)} -m pip install -r requirements.txt --upgrade-strategy only-if-needed", cwd=project_dir)
run(f"{q(sys.executable)} -m pip install kaggle --upgrade-strategy only-if-needed")

kaggle_config_dir = configure_kaggle(KAGGLE_USERNAME, KAGGLE_KEY, Path("/content"))

base_config = project_dir / "configs" / "animefaces_512.json"
config = json.loads(base_config.read_text(encoding="utf-8"))
config["device"] = "cuda"
config["dataset_zip"] = str(dataset_zip)
config["run_root"] = str(run_root)
config["kimg"] = int(KIMG)
config["batch_size"] = int(BATCH_SIZE)
config["snap"] = int(SNAP)
config["augpipe"] = AUGPIPE
config["resume"] = RESUME
config["allow_tf32"] = True
config["nhwc"] = True
config["workers"] = 8

runtime_config = config_dir / "stylegan2_ada_animefaces_512_config.json"
runtime_config.write_text(json.dumps(config, indent=2), encoding="utf-8")
print("Runtime config:", runtime_config)
print("Dataset zip:", dataset_zip)
print("Run root:", run_root)

env = os.environ.copy()
env.update({
    "CONFIG": str(runtime_config),
    "DATASET_ZIP": str(dataset_zip),
    "DOWNLOAD_DIR": str(download_dir),
    "RUN_TRAIN": "1",
    "STYLEGAN_DEVICE": "cuda",
    "KAGGLE_CONFIG_DIR": str(kaggle_config_dir),
})

run("bash scripts/prepare_animefaces_512.sh", cwd=project_dir, env=env)

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
archive_base = Path("/content") / f"stylegan2_ada_animefaces_512_{timestamp}"
archive_path = shutil.make_archive(str(archive_base), "zip", root_dir=content_root)
drive_backup_dir.mkdir(parents=True, exist_ok=True)
drive_archive = drive_backup_dir / Path(archive_path).name
shutil.copy2(archive_path, drive_archive)

print("Training complete.")
print("Local archive:", archive_path)
print("Copied archive:", drive_archive)
print("Local runs:", run_root / "stylegan2_ada" / "animefaces_512")